# NLP for Shakespeare sonet

In this project, we aim to extend a given sonnet using Natural Language Processing (NLP). To achieve this, we train an LSTM-based model on actual sonnet text, enabling it to learn the style, structure, and rhythm of the original work and generate new lines that seamlessly continue the poem.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
with open('Data/shakespeare.txt','r', encoding='utf8') as f:
    text = f.read()

In [3]:
type(text)

str

In [4]:
print(text[:1000])


                     1
  From fairest creatures we desire increase,
  That thereby beauty's rose might never die,
  But as the riper should by time decease,
  His tender heir might bear his memory:
  But thou contracted to thine own bright eyes,
  Feed'st thy light's flame with self-substantial fuel,
  Making a famine where abundance lies,
  Thy self thy foe, to thy sweet self too cruel:
  Thou that art now the world's fresh ornament,
  And only herald to the gaudy spring,
  Within thine own bud buriest thy content,
  And tender churl mak'st waste in niggarding:
    Pity the world, or else this glutton be,
    To eat the world's due, by the grave and thee.


                     2
  When forty winters shall besiege thy brow,
  And dig deep trenches in thy beauty's field,
  Thy youth's proud livery so gazed on now,
  Will be a tattered weed of small worth held:  
  Then being asked, where all thy beauty lies,
  Where all the treasure of thy lusty days;
  To say within thine own deep su

In [5]:
len(text)

5445609

In [6]:
## Unique characters
all_characters = set(text)

In [7]:
len(all_characters)

84

In [8]:
# Characters --> Index 
encoder = {char:idx for idx,char in enumerate(all_characters)} 

In [9]:
# Index --> Characters
decoder = {idx:char for idx,char in enumerate(all_characters)}

In [10]:
# Let's encode the whole text
encoded_text = np.array([encoder[char] for char in text])

In [11]:
encoded_text[:500]

array([34, 82, 82, 82, 82, 82, 82, 82, 82, 82, 82, 82, 82, 82, 82, 82, 82,
       82, 82, 82, 82, 82, 48, 34, 82, 82, 77, 10,  3, 29, 82, 26, 21, 69,
       10, 30, 60, 63, 82, 55, 10, 30, 21, 63, 27, 10, 30, 60, 82, 71, 30,
       82, 53, 30, 60, 69, 10, 30, 82, 69, 52, 55, 10, 30, 21, 60, 30,  2,
       34, 82, 82,  4, 57, 21, 63, 82, 63, 57, 30, 10, 30, 66,  0, 82, 66,
       30, 21, 27, 63,  0,  8, 60, 82, 10,  3, 60, 30, 82, 29, 69,  5, 57,
       63, 82, 52, 30, 61, 30, 10, 82, 53, 69, 30,  2, 34, 82, 82, 68, 27,
       63, 82, 21, 60, 82, 63, 57, 30, 82, 10, 69,  6, 30, 10, 82, 60, 57,
        3, 27, 78, 53, 82, 66,  0, 82, 63, 69, 29, 30, 82, 53, 30, 55, 30,
       21, 60, 30,  2, 34, 82, 82, 74, 69, 60, 82, 63, 30, 52, 53, 30, 10,
       82, 57, 30, 69, 10, 82, 29, 69,  5, 57, 63, 82, 66, 30, 21, 10, 82,
       57, 69, 60, 82, 29, 30, 29,  3, 10,  0, 33, 34, 82, 82, 68, 27, 63,
       82, 63, 57,  3, 27, 82, 55,  3, 52, 63, 10, 21, 55, 63, 30, 53, 82,
       63,  3, 82, 63, 57

In [12]:
decoder[82]

' '

In [13]:
np.arange(6)

array([0, 1, 2, 3, 4, 5])

In [14]:
def one_hot_encoder(encoded_text, num_uni_chars):
    one_hot = np.zeros((encoded_text.size,num_uni_chars))
    # Convert data type for later use with pytorch (errors if we dont!)
    one_hot = one_hot.astype(np.float32)
    one_hot[np.arange(one_hot.shape[0]),encoded_text.flatten()] = 1.0
    one_hot = one_hot.reshape((*encoded_text.shape,num_uni_chars))
    return one_hot

In [15]:
# check
arr = np.array([1,2,2])
one_hot_encoder(arr,3)

array([[0., 1., 0.],
       [0., 0., 1.],
       [0., 0., 1.]], dtype=float32)

In [16]:
def generate_batches(encoded_text, samp_per_batch=10, seq_len=50):
    
    '''
    Generate (using yield) batches for training.
    
    X: Encoded Text of length seq_len
    Y: Encoded Text shifted by one
    
    Example:
    
    X:
    
    [[1 2 3]]
    
    Y:
    
    [[ 2 3 4]]
    
    encoded_text : Complete Encoded Text to make batches from
    batch_size : Number of samples per batch
    seq_len : Length of character sequence
       
    '''
    
    # Total number of characters per batch
    # Example: If samp_per_batch is 2 and seq_len is 50, then 100
    # characters come out per batch.
    char_per_batch = samp_per_batch * seq_len
    
    
    # Number of batches available to make
    # Use int() to round to nearest integer
    num_batches_avail = int(len(encoded_text)/char_per_batch)
    
    # Cut off end of encoded_text that
    # won't fit evenly into a batch
    encoded_text = encoded_text[:num_batches_avail * char_per_batch]
    
    
    # Reshape text into rows the size of a batch
    encoded_text = encoded_text.reshape((samp_per_batch, -1))
    

    # Go through each row in array.
    for n in range(0, encoded_text.shape[1], seq_len):
        
        # Grab feature characters
        x = encoded_text[:, n:n+seq_len]
        
        # y is the target shifted over by 1
        y = np.zeros_like(x)
       
        #
        try:
            y[:, :-1] = x[:, 1:]
            y[:, -1]  = encoded_text[:, n+seq_len]
            
        # FOR POTENTIAL INDEXING ERROR AT THE END    
        except:
            y[:, :-1] = x[:, 1:]
            y[:, -1] = encoded_text[:, 0]
            
        yield x, y

In [17]:
sample_text = np.arange(20)
sample_text

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19])

In [18]:
batch_generator = generate_batches(sample_text,samp_per_batch=2,seq_len=5)
x,y = next(batch_generator)

** Here, whole x is first batch. And correspondingly y is label batch for x respectively.

In [19]:
x

array([[ 0,  1,  2,  3,  4],
       [10, 11, 12, 13, 14]])

In [20]:
y

array([[ 1,  2,  3,  4,  5],
       [11, 12, 13, 14, 15]])

In [21]:
x,y = next(batch_generator)

In [22]:
x

array([[ 5,  6,  7,  8,  9],
       [15, 16, 17, 18, 19]])

In [23]:
y

array([[ 6,  7,  8,  9,  0],
       [16, 17, 18, 19, 10]])

## LSTM Model

In [24]:
class CharModel(nn.Module):
    
    def __init__(self, all_chars, num_hidden=256, num_layers=4,drop_prob=0.5,use_gpu=False):
        
        
        # SET UP ATTRIBUTES
        super().__init__()
        self.drop_prob = drop_prob
        self.num_layers = num_layers
        self.num_hidden = num_hidden
        self.use_gpu = use_gpu
        
        #CHARACTER SET, ENCODER, and DECODER
        self.all_chars = all_chars
        self.decoder = dict(enumerate(all_chars))
        self.encoder = {char: ind for ind,char in decoder.items()}
        
        
        self.lstm = nn.LSTM(len(self.all_chars), num_hidden, num_layers, dropout=drop_prob, batch_first=True)
        
        self.dropout = nn.Dropout(drop_prob)
        
        self.fc_linear = nn.Linear(num_hidden, len(self.all_chars))
      
    
    def forward(self, x, hidden):
                  
        
        lstm_output, hidden = self.lstm(x, hidden)
        
        
        drop_output = self.dropout(lstm_output)
        
        drop_output = drop_output.contiguous().view(-1, self.num_hidden)
        
        
        final_out = self.fc_linear(drop_output)
        
        
        return final_out, hidden
    
    
    def hidden_state(self, batch_size):
        '''
        Used as separate method to account for both GPU and CPU users.
        '''
        
        if self.use_gpu:
            
            hidden = (torch.zeros(self.num_layers,batch_size,self.num_hidden).cuda(),
                     torch.zeros(self.num_layers,batch_size,self.num_hidden).cuda())
        else:
            hidden = (torch.zeros(self.num_layers,batch_size,self.num_hidden),
                     torch.zeros(self.num_layers,batch_size,self.num_hidden))
        
        return hidden
        

In [25]:
model = CharModel(all_chars=all_characters,
                 num_hidden=512,
                  num_layers=3,
                 drop_prob=0.5)

In [26]:
model

CharModel(
  (lstm): LSTM(84, 512, num_layers=3, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc_linear): Linear(in_features=512, out_features=84, bias=True)
)

In [27]:
total_param  = []
for p in model.parameters():
    total_param.append(int(p.numel()))

In [28]:
sum(total_param)

5470292

In [29]:
len(encoded_text)

5445609

In [30]:
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
criterion = nn.CrossEntropyLoss()

## Training Data and Validation Data

In [31]:
# percentage of data to be used for training
train_percent = 0.1

In [32]:
train_ind = int(len(encoded_text) * (train_percent))

In [33]:
train_data = encoded_text[:train_ind]
val_data = encoded_text[train_ind:]

## Training the network

In [34]:
epochs = 20
batch_size = 100

seq_len = 100
tracker = 0             # For printing
num_char = max(encoded_text)+1     # +1 because indexing starts at 0

In [35]:
model.train()

for i in range(epochs):
    hidden = model.hidden_state(batch_size)

    for x,y in generate_batches(train_data,batch_size,seq_len):
        tracker += 1

        ## Input
        x = one_hot_encoder(x,num_char)
        inputs = torch.from_numpy(x)
        
        ## Label
        targets = torch.from_numpy(y)

        hidden = tuple([state.data for state in hidden])

        model.zero_grad()

        ## Calculating the LSTM outputs and loss
        lstm_output, hidden = model.forward(inputs, hidden)
        loss = criterion(lstm_output,targets.view(batch_size*seq_len).long())
        loss.backward()
        
        ## Clipping the gradients, if necessary
        nn.utils.clip_grad_norm_(model.parameters(),max_norm=5)
        optimizer.step()

        ## Validation at every 25th step
        if tracker % 25 == 0:
            val_hidden = model.hidden_state(batch_size)
            val_losses = []
            model.eval()

            for x,y in generate_batches(val_data, batch_size, seq_len):
                
                # Input
                x = one_hot_encoder(x,num_char)
                inputs = torch.from_numpy(x)
            
                # Label
                targets = torch.from_numpy(y)

                val_hidden = tuple([state.data for state in val_hidden])
                lstm_output, val_hidden = model.forward(inputs, val_hidden)
                val_loss = criterion(lstm_output,targets.view(batch_size*seq_len).long())
                
                val_losses.append(val_loss.item())

             # Reset to training model after val for loop
            model.train()
            
            print(f"Epoch: {i} Step: {tracker} Val Loss: {val_loss.item()}")

Epoch: 0 Step: 25 Val Loss: 3.1924662590026855
Epoch: 0 Step: 50 Val Loss: 3.1838431358337402
Epoch: 1 Step: 75 Val Loss: 3.177639961242676
Epoch: 1 Step: 100 Val Loss: 3.068591356277466
Epoch: 2 Step: 125 Val Loss: 2.9611637592315674
Epoch: 2 Step: 150 Val Loss: 2.7960877418518066
Epoch: 3 Step: 175 Val Loss: 2.678481340408325
Epoch: 3 Step: 200 Val Loss: 2.5655112266540527
Epoch: 4 Step: 225 Val Loss: 2.4602081775665283
Epoch: 4 Step: 250 Val Loss: 2.341935396194458
Epoch: 5 Step: 275 Val Loss: 2.2703254222869873
Epoch: 5 Step: 300 Val Loss: 2.2006547451019287
Epoch: 6 Step: 325 Val Loss: 2.151087999343872
Epoch: 6 Step: 350 Val Loss: 2.1151230335235596
Epoch: 6 Step: 375 Val Loss: 2.067894458770752
Epoch: 7 Step: 400 Val Loss: 2.043851137161255
Epoch: 7 Step: 425 Val Loss: 2.0163211822509766
Epoch: 8 Step: 450 Val Loss: 1.9886199235916138
Epoch: 8 Step: 475 Val Loss: 1.9660916328430176
Epoch: 9 Step: 500 Val Loss: 1.9480890035629272
Epoch: 9 Step: 525 Val Loss: 1.9315201044082642
Ep

## Saving the model

In [36]:
torch.save(model.state_dict(),'Hidden512_layer3_Shakespeare.net')

In [37]:
def predict_next_char(model, char, hidden=None, k=1):
        
        # Encode raw letters with model
        encoded_text = model.encoder[char]
        
        # set as numpy array for one hot encoding
        # NOTE THE [[ ]] dimensions!!
        encoded_text = np.array([[encoded_text]])
        
        # One hot encoding
        encoded_text = one_hot_encoder(encoded_text, len(model.all_chars))
        
        # Convert to Tensor
        inputs = torch.from_numpy(encoded_text)
        
        # Check for CPU
        if(model.use_gpu):
            inputs = inputs.cuda()
        
        
        # Grab hidden states
        hidden = tuple([state.data for state in hidden])
        
        
        # Run model and get predicted output
        lstm_out, hidden = model(inputs, hidden)

        
        # Convert lstm_out to probabilities
        probs = F.softmax(lstm_out, dim=1).data
        
        
        
        if(model.use_gpu):
            # move back to CPU to use with numpy
            probs = probs.cpu()
        
        
        # k determines how many characters to consider
        # for our probability choice.
        # https://pytorch.org/docs/stable/torch.html#torch.topk
        
        # Return k largest probabilities in tensor
        probs, index_positions = probs.topk(k)
        
        
        index_positions = index_positions.numpy().squeeze()
        
        # Create array of probabilities
        probs = probs.numpy().flatten()
        
        # Convert to probabilities per index
        probs = probs/probs.sum()
        
        # randomly choose a character based on probabilities
        char = np.random.choice(index_positions, p=probs)
       
        # return the encoded value of the predicted char and the hidden state
        return model.decoder[char], hidden

In [38]:
def generate_text(model, size, seed='The', k=1):
        
      
    
    # CHECK FOR GPU
    if(model.use_gpu):
        model.cuda()
    else:
        model.cpu()
    
    # Evaluation mode
    model.eval()
    
    # begin output from initial seed
    output_chars = [c for c in seed]
    
    # intiate hidden state
    hidden = model.hidden_state(1)
    
    # predict the next character for every character in seed
    for char in seed:
        char, hidden = predict_next_char(model, char, hidden, k=k)
    
    # add initial characters to output
    output_chars.append(char)
    
    # Now generate for size requested
    for i in range(size):
        
        # predict based off very last letter in output_chars
        char, hidden = predict_next_char(model, output_chars[-1], hidden, k=k)
        
        # add predicted character
        output_chars.append(char)
    
    # return string of predicted text
    return ''.join(output_chars)

In [39]:
print(generate_text(model, 1000, seed='The ', k=3))

The stard,
    And some to make her breathed to him to my lord,
    And she that strenged thou shalt stor hath thou art show to the weet
    That she that thou art stronged. To the both that were
    Which strengest on the streets.
  CLEOPATRA. I shall send me to thing and so madring thee.
  ANTONY. I am this shall shanl the strengest store.
    I she so make the world best tree thou art.
    The waste thou she stands our beaunge in me the word,
    That's my love and the sent of thy poor stance
    To the shept as that she then to the stard.
    The seets on the stands and sendess and heart,
    The worlds to star the wines ther should be son to her.
    That have the bott that sent a strong and strests
    To the brangs of his stors. I would see, she have street
    And that I have should by man see and see hate too,
    The warter stange of my. Thou should I
    The bears thou shouldst show thee and shourss but his beanty,
    And to the strange that have breakst to the store.


   